In [2]:
import pandas as pd
import pm4py
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load your data
df = pd.read_csv('EDHospital.csv')

# Convert timestamp and format for PM4Py
df['Complete Timestamp'] = pd.to_datetime(df['Complete Timestamp'])
event_log = pm4py.format_dataframe(df, 
                                  case_id='Case ID', 
                                  activity_key='Activity', 
                                  timestamp_key='Complete Timestamp')

print("Data loaded successfully!")
print(f"Number of cases: {len(event_log['Case ID'].unique())}")
print(f"Number of events: {len(event_log)}")

Data loaded successfully!
Number of cases: 55079
Number of events: 461839


In [3]:
print("=== PART 1: LOG EXPLORATION (PURE PM4PY) ===")

# Convert to PM4Py EventLog object first (if not already done)
if isinstance(event_log, pd.DataFrame):
    from pm4py.objects.conversion.log import converter as log_converter
    log = log_converter.apply(event_log)
else:
    log = event_log

# 1.1 Main activities and their frequencies using PM4Py
from pm4py.statistics.attributes.log import get as attributes_get

# Get activity frequencies
activity_counts = attributes_get.get_attribute_values(log, "concept:name")
print("\n1.1 Main Activities and Frequencies:")
for activity, count in sorted(activity_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {activity}: {count} occurrences")

# Most common activity
most_common_activity = max(activity_counts.items(), key=lambda x: x[1])
print(f"\nMost common activity: '{most_common_activity[0]}' ({most_common_activity[1]} occurrences)")

# 1.2 Variants analysis using PM4Py
from pm4py.statistics.traces.generic.log import case_statistics

# Get variant statistics
variant_stats = case_statistics.get_variant_statistics(log)
print(f"\n1.2 Total number of variants (unique paths): {len(variant_stats)}")

# Most common variant
most_common_variant = max(variant_stats, key=lambda x: x['count'])
print(f"\nMost common variant (frequency: {most_common_variant['count']} cases):")
print(" -> ".join(most_common_variant['variant']))

# Show top 5 variants
print("\nTop 5 most frequent variants:")
for i, variant in enumerate(sorted(variant_stats, key=lambda x: x['count'], reverse=True)[:5]):
    print(f"{i+1}. Frequency: {variant['count']} - {' -> '.join(variant['variant'])}")

=== PART 1: LOG EXPLORATION (PURE PM4PY) ===

1.1 Main Activities and Frequencies:
  Reception: 55079 occurrences
  Discharge: 55079 occurrences
  Doctor admission: 54871 occurrences
  Nurse admission: 54300 occurrences
  Discharge (Doctor): 54213 occurrences
  Last lab tests results: 41268 occurrences
  Order Imaging test: 29730 occurrences
  Order blood tests: 28043 occurrences
  Imaging: 27824 occurrences
  Additional vitals: 25188 occurrences
  Order Consultant: 15192 occurrences
  Consultant: 12954 occurrences
  Imaging decrypting: 7110 occurrences
  Order external exams: 506 occurrences
  External exams: 482 occurrences

Most common activity: 'Reception' (55079 occurrences)

1.2 Total number of variants (unique paths): 9537

Most common variant (frequency: 7994 cases):
Reception -> Nurse admission -> Order blood tests -> Last lab tests results -> Doctor admission -> Discharge (Doctor) -> Discharge

Top 5 most frequent variants:
1. Frequency: 7994 - Reception -> Nurse admission ->

In [9]:
print("\n=== PART 2: TIME PERSPECTIVE (STABLE PM4PY) ===")

import numpy as np
import time
start_time = time.time()

# Convert to PM4Py EventLog object
from pm4py.objects.conversion.log import converter as log_converter
log = log_converter.apply(event_log)

# 2.1 Case duration statistics using PM4Py
from pm4py.statistics.traces.generic.log import case_statistics

case_durations = case_statistics.get_all_case_durations(log)
print(f"2.1 Case Duration Statistics:")
print(f"  Average: {pd.Timedelta(seconds=np.mean(case_durations))}")
print(f"  Shortest: {pd.Timedelta(seconds=np.min(case_durations))}")
print(f"  Longest: {pd.Timedelta(seconds=np.max(case_durations))}")
print(f"  Median: {pd.Timedelta(seconds=np.median(case_durations))}")
print(f"  Cases: {len(case_durations)}")

# 2.2 Activity analysis - using basic PM4Py functions
from pm4py.statistics.attributes.log import get as attributes_get

activities = attributes_get.get_attribute_values(log, "concept:name")
print(f"\n2.2 Activity Analysis:")
print(f"  Total unique activities: {len(activities)}")
print(f"  Most frequent activity: {max(activities.items(), key=lambda x: x[1])}")

# 2.3 Variant analysis with durations
print(f"\n2.3 Path Duration Analysis:")

from pm4py.algo.filtering.log.variants import variants_filter
from pm4py.statistics.traces.generic.log import case_statistics

variants = variants_filter.get_variants(log)
variant_stats = case_statistics.get_variant_statistics(log)

print(f"Total unique paths (variants): {len(variants)}")

# Show top 3 variants with their durations
top_variants = sorted(variant_stats, key=lambda x: x['count'], reverse=True)[:3]

for i, variant_info in enumerate(top_variants):
    variant = variant_info['variant']
    variant_log = variants_filter.apply(log, [variant])
    durations = case_statistics.get_all_case_durations(variant_log)
    avg_duration = np.mean(durations) if durations else 0
    
    print(f"\nVariant {i+1} (Frequency: {variant_info['count']} cases):")
    print(f"  Average duration: {pd.Timedelta(seconds=avg_duration)}")
    print(f"  Path: {' -> '.join(variant[:5])}{'...' if len(variant) > 5 else ''}")

# 2.4 Performance analysis using basic statistics
print(f"\n2.4 Performance Analysis:")

# Calculate percentiles for case durations
p25 = np.percentile(case_durations, 25)
p50 = np.percentile(case_durations, 50)
p75 = np.percentile(case_durations, 75)
p90 = np.percentile(case_durations, 90)

print("Case duration percentiles:")
print(f"  25th: {pd.Timedelta(seconds=p25)}")
print(f"  50th (median): {pd.Timedelta(seconds=p50)}")
print(f"  75th: {pd.Timedelta(seconds=p75)}")
print(f"  90th: {pd.Timedelta(seconds=p90)}")

# 2.5 Process insights
print(f"\n2.5 Process Insights:")

# Identify potential bottlenecks based on variant analysis
if top_variants:
    longest_variant = max(top_variants, 
                         key=lambda x: np.mean(case_statistics.get_all_case_durations(
                             variants_filter.apply(log, [x['variant']])
                         )))
    
    shortest_variant = min(top_variants, 
                          key=lambda x: np.mean(case_statistics.get_all_case_durations(
                              variants_filter.apply(log, [x['variant']])
                          )))
    
    print(f"  Among top variants:")
    print(f"  - Longest path: {pd.Timedelta(seconds=np.mean(case_statistics.get_all_case_durations(variants_filter.apply(log, [longest_variant['variant']]))))}")
    print(f"  - Shortest path: {pd.Timedelta(seconds=np.mean(case_statistics.get_all_case_durations(variants_filter.apply(log, [shortest_variant['variant']]))))}")

# Throughput analysis
print(f"\nThroughput insights:")
print(f"  Average processing time: {pd.Timedelta(seconds=np.mean(case_durations))}")
print(f"  Process efficiency range: {pd.Timedelta(seconds=p25)} to {pd.Timedelta(seconds=p75)}")
print(f"  Extreme cases: {len([d for d in case_durations if d > p90])} cases above 90th percentile")

end_time = time.time()
print(f"\nPM4Py analysis completed in {end_time - start_time:.1f} seconds")


=== PART 2: TIME PERSPECTIVE (STABLE PM4PY) ===
2.1 Case Duration Statistics:
  Average: 0 days 07:56:46.159879445
  Shortest: 0 days 00:01:00
  Longest: 221 days 13:46:00
  Median: 0 days 05:24:00
  Cases: 55079

2.2 Activity Analysis:
  Total unique activities: 15
  Most frequent activity: ('Reception', 55079)

2.3 Path Duration Analysis:
Total unique paths (variants): 9537

Variant 1 (Frequency: 7994 cases):
  Average duration: 0 days 08:58:37.535651738
  Path: Reception -> Nurse admission -> Order blood tests -> Last lab tests results -> Doctor admission...

Variant 2 (Frequency: 5015 cases):
  Average duration: 0 days 03:53:31.214356929
  Path: Reception -> Nurse admission -> Doctor admission -> Discharge (Doctor) -> Discharge

Variant 3 (Frequency: 4613 cases):
  Average duration: 0 days 08:12:43.585519184
  Path: Reception -> Nurse admission -> Order blood tests -> Doctor admission -> Last lab tests results...

2.4 Performance Analysis:
Case duration percentiles:
  25th: 0 days

In [11]:
print("\n=== PART 2: TIME PERSPECTIVE (ACCURATE CALCULATION) ===")

import time
start_time = time.time()

# Convert to PM4Py log
from pm4py.objects.conversion.log import converter as log_converter
log = log_converter.apply(event_log)

# 2.1 Case duration statistics - but let's investigate the outliers
from pm4py.statistics.traces.generic.log import case_statistics
case_durations = case_statistics.get_all_case_durations(log)

print(f"2.1 Case Duration Analysis:")
print(f"  Raw calculation (first to last event):")
print(f"    Average: {pd.Timedelta(seconds=np.mean(case_durations))}")
print(f"    Shortest: {pd.Timedelta(seconds=np.min(case_durations))}")
print(f"    Longest: {pd.Timedelta(seconds=np.max(case_durations))}")
print(f"    Median: {pd.Timedelta(seconds=np.median(case_durations))}")

# Let's check if there are unrealistic durations
SECONDS_IN_DAY = 86400
realistic_threshold = 30 * SECONDS_IN_DAY  # 30 days max for ED

realistic_durations = [d for d in case_durations if d <= realistic_threshold]
outliers = [d for d in case_durations if d > realistic_threshold]

print(f"\n  Data Quality Check:")
print(f"    Realistic cases (<30 days): {len(realistic_durations)}")
print(f"    Outliers (>30 days): {len(outliers)}")
print(f"    Outlier durations: {[f'{d/SECONDS_IN_DAY:.1f} days' for d in outliers]}")

if realistic_durations:
    print(f"\n  Realistic case statistics:")
    print(f"    Average: {pd.Timedelta(seconds=np.mean(realistic_durations))}")
    print(f"    Longest realistic: {pd.Timedelta(seconds=np.max(realistic_durations))}")
    print(f"    95th percentile: {pd.Timedelta(seconds=np.percentile(realistic_durations, 95))}")

# 2.2 Let's calculate "active time" similar to your @@flow_time
print(f"\n2.2 Active Time Analysis (like @@flow_time):")

def calculate_active_time(log):
    """Calculate sum of time between consecutive activities within cases"""
    active_times = []
    
    for trace in log:
        if len(trace) > 1:
            # Sort events by timestamp
            sorted_events = sorted(trace, key=lambda x: x['time:timestamp'])
            
            # Calculate sum of time between consecutive activities
            active_time = 0
            for i in range(len(sorted_events) - 1):
                time_diff = (sorted_events[i + 1]['time:timestamp'] - sorted_events[i]['time:timestamp']).total_seconds()
                # Only count reasonable gaps (less than 1 week)
                if time_diff < 7 * SECONDS_IN_DAY:
                    active_time += time_diff
            
            active_times.append(active_time)
    
    return active_times

active_times = calculate_active_time(log)

if active_times:
    print(f"  Active time statistics:")
    print(f"    Average active time: {pd.Timedelta(seconds=np.mean(active_times))}")
    print(f"    Shortest active time: {pd.Timedelta(seconds=np.min(active_times))}")
    print(f"    Longest active time: {pd.Timedelta(seconds=np.max(active_times))}")
    print(f"    Median active time: {pd.Timedelta(seconds=np.median(active_times))}")
    
    # Convert longest active time to days for comparison
    longest_active_days = np.max(active_times) / SECONDS_IN_DAY
    print(f"    Longest active time in days: {longest_active_days:.1f} days")

# 2.3 Variant analysis with both duration types
print(f"\n2.3 Variant Duration Comparison:")

from pm4py.algo.filtering.log.variants import variants_filter
from pm4py.statistics.traces.generic.log import case_statistics

variants = variants_filter.get_variants(log)
top_variants = sorted(variants.items(), key=lambda x: len(x[1]), reverse=True)[:3]

for i, (variant, cases) in enumerate(top_variants):
    variant_log = variants_filter.apply(log, [variant])
    
    # Total duration
    total_durations = case_statistics.get_all_case_durations(variant_log)
    
    # Active duration approximation
    active_durations = calculate_active_time(variant_log)
    
    print(f"\nVariant {i+1} ({len(cases)} cases):")
    print(f"  {' -> '.join(variant[:4])}{'...' if len(variant) > 4 else ''}")
    if total_durations:
        print(f"  Total duration: {pd.Timedelta(seconds=np.mean(total_durations))}")
    if active_durations:
        active_days = np.mean(active_durations) / SECONDS_IN_DAY
        print(f"  Active time: {pd.Timedelta(seconds=np.mean(active_durations))} ({active_days:.1f} days)")

# 2.4 Summary
print(f"\n2.4 Key Insights:")
print(f"  • The 221-day 'case' is likely a data quality issue")
print(f"  • Real ED cases typically complete within hours or days")
print(f"  • Your @@flow_time calculation (55 days) is more realistic")
print(f"  • Active time excludes long waiting periods between activities")

end_time = time.time()
print(f"\nAccurate analysis completed in {end_time - start_time:.1f} seconds")


=== PART 2: TIME PERSPECTIVE (ACCURATE CALCULATION) ===
2.1 Case Duration Analysis:
  Raw calculation (first to last event):
    Average: 0 days 07:56:46.159879445
    Shortest: 0 days 00:01:00
    Longest: 221 days 13:46:00
    Median: 0 days 05:24:00

  Data Quality Check:
    Realistic cases (<30 days): 55072
    Outliers (>30 days): 7
    Outlier durations: ['37.8 days', '55.9 days', '85.3 days', '116.1 days', '118.9 days', '119.4 days', '221.6 days']

  Realistic case statistics:
    Average: 0 days 07:37:05.327571179
    Longest realistic: 24 days 08:48:00
    95th percentile: 0 days 22:03:26.999999999

2.2 Active Time Analysis (like @@flow_time):
  Active time statistics:
    Average active time: 0 days 07:33:40.756731240
    Shortest active time: 0 days 00:01:00
    Longest active time: 20 days 10:30:00
    Median active time: 0 days 05:24:00
    Longest active time in days: 20.4 days

2.3 Variant Duration Comparison:

Variant 1 (7994 cases):
  Reception -> Nurse admission -> 

In [12]:
print("\n=== AVERAGE DELAYS ANALYSIS ===")

def calculate_average_delays(log):
    """Calculate average waiting times between consecutive activities"""
    transition_delays = {}
    
    for trace in log:
        if len(trace) > 1:
            sorted_events = sorted(trace, key=lambda x: x['time:timestamp'])
            
            for i in range(len(sorted_events) - 1):
                current_activity = sorted_events[i]['concept:name']
                next_activity = sorted_events[i + 1]['concept:name']
                transition = f"{current_activity} -> {next_activity}"
                
                delay = (sorted_events[i + 1]['time:timestamp'] - sorted_events[i]['time:timestamp']).total_seconds()
                
                if transition not in transition_delays:
                    transition_delays[transition] = []
                transition_delays[transition].append(delay)
    
    # Calculate averages and filter for meaningful transitions
    avg_delays = {}
    for transition, delays in transition_delays.items():
        if len(delays) >= 10:  # Only include transitions with sufficient data
            avg_delays[transition] = np.mean(delays)
    
    return avg_delays

# Calculate average delays
avg_delays = calculate_average_delays(log)

print("Key Transition Delays (Average Waiting Times):")

# Focus on important transitions
key_transitions = [
    "Reception -> Nurse admission",
    "Nurse admission -> Doctor admission", 
    "Doctor admission -> Discharge (Doctor)",
    "Discharge (Doctor) -> Discharge",
    "Order blood tests -> Last lab tests results",
    "Order Imaging test -> Imaging"
]

for transition in key_transitions:
    if transition in avg_delays:
        print(f"  {transition}: {pd.Timedelta(seconds=avg_delays[transition])}")

# Also show top 5 longest delays overall
print(f"\nTop 5 Longest Average Delays:")
sorted_delays = sorted(avg_delays.items(), key=lambda x: x[1], reverse=True)[:5]
for transition, delay in sorted_delays:
    print(f"  {transition}: {pd.Timedelta(seconds=delay)}")

print(f"\nTop 5 Shortest Average Delays:")
sorted_fast = sorted(avg_delays.items(), key=lambda x: x[1])[:5]
for transition, delay in sorted_fast:
    print(f"  {transition}: {pd.Timedelta(seconds=delay)}")


=== AVERAGE DELAYS ANALYSIS ===
Key Transition Delays (Average Waiting Times):
  Reception -> Nurse admission: 0 days 00:30:53.587605232
  Nurse admission -> Doctor admission: 0 days 00:38:02.369215291
  Doctor admission -> Discharge (Doctor): 0 days 03:18:50.566838995
  Discharge (Doctor) -> Discharge: 0 days 02:01:41.659087575
  Order blood tests -> Last lab tests results: 0 days 00:27:11.423332279
  Order Imaging test -> Imaging: 0 days 00:48:19.766536964

Top 5 Longest Average Delays:
  Discharge (Doctor) -> Nurse admission: 2 days 04:56:48.699551569
  Imaging decrypting -> Nurse admission: 0 days 18:13:51.428571428
  Doctor admission -> Discharge: 0 days 10:44:47.191011235
  Consultant -> Nurse admission: 0 days 08:50:21
  External exams -> Additional vitals: 0 days 07:06:24

Top 5 Shortest Average Delays:
  Nurse admission -> Order blood tests: 0 days 00:01:15.530477448
  Nurse admission -> Additional vitals: 0 days 00:02:42.466091245
  Order blood tests -> Nurse admission: 0 da

In [14]:
print("\n=== PART 3: INFREQUENT VARIANTS (SIMPLE PM4PY) ===")

from pm4py.statistics.traces.generic.log import case_statistics
from pm4py.algo.filtering.log.variants import variants_filter

# Get variant statistics
variant_stats = case_statistics.get_variant_statistics(log)

# Basic analysis
infrequent = [v for v in variant_stats if v['count'] == 1]
frequent = [v for v in variant_stats if v['count'] > 1]

print(f"Basic Statistics:")
print(f"  {len(infrequent)} infrequent variants (occur once)")
print(f"  {len(frequent)} frequent variants (occur multiple times)")

# Quick duration comparison
def get_avg_duration(variants_list):
    """Get average duration for a list of variants using PM4Py"""
    if not variants_list:
        return 0
    # Use first 10 variants for quick comparison
    durations = []
    for v in variants_list[:10]:
        variant_log = variants_filter.apply(log, [v['variant']])
        durs = case_statistics.get_all_case_durations(variant_log)
        if durs:
            durations.append(np.mean(durs))
    return np.mean(durations) if durations else 0

avg_infreq = get_avg_duration(infrequent)
avg_freq = get_avg_duration(frequent)

print(f"\nQuick Duration Comparison:")
print(f"  Frequent variants: {pd.Timedelta(seconds=avg_freq)}")
print(f"  Infrequent variants: {pd.Timedelta(seconds=avg_infreq)}")

if avg_infreq > avg_freq:
    print(f"Infrequent cases are LONGER")
else:
    print(f"Infrequent cases are SHORTER")

print(f"\nAnswer: Infrequent variants are {'LONGER' if avg_infreq > avg_freq else 'SHORTER'} than frequent variants")


=== PART 3: INFREQUENT VARIANTS (SIMPLE PM4PY) ===
Basic Statistics:
  7703 infrequent variants (occur once)
  1834 frequent variants (occur multiple times)

Quick Duration Comparison:
  Frequent variants: 0 days 06:11:09.316833407
  Infrequent variants: 0 days 08:05:24
Infrequent cases are LONGER

Answer: Infrequent variants are LONGER than frequent variants


In [15]:
print("\n=== PART 4: PROCESS DISCOVERY ALGORITHMS ===")

# Convert to PM4Py EventLog object first
from pm4py.objects.conversion.log import converter as log_converter
log = log_converter.apply(event_log)

print("Applying three process discovery algorithms...")

# 4.1 Alpha Miner (using Alpha+ as shown in professor's code)
print("\n--- ALPHA MINER ---")
try:
    # Try different alpha variations
    net_alpha_plus, initial_marking_alpha, final_marking_alpha = pm4py.discover_petri_net_alpha_plus(log)
    print("✓ Alpha+ Miner completed successfully")
    
    # Visualize Alpha Miner Petri net
    pm4py.view_petri_net(net_alpha_plus, initial_marking_alpha, final_marking_alpha)
    
    # Convert to BPMN
    bpmn_alpha = pm4py.convert_to_bpmn(net_alpha_plus, initial_marking_alpha, final_marking_alpha)
    pm4py.view_bpmn(bpmn_alpha)
    print("✓ Alpha Miner BPMN visualization opened")
    
except Exception as e:
    print(f"✗ Alpha+ Miner failed: {e}")
    # Try regular Alpha miner
    try:
        from pm4py.algo.discovery.alpha import algorithm as alpha_miner
        net_alpha_plus, initial_marking_alpha, final_marking_alpha = alpha_miner.apply(log)
        print("✓ Regular Alpha Miner completed successfully")
        pm4py.view_petri_net(net_alpha_plus, initial_marking_alpha, final_marking_alpha)
    except Exception as e2:
        print(f"✗ All Alpha variations failed: {e2}")

# 4.2 Heuristic Miner (using professor's parameters)
print("\n--- HEURISTIC MINER ---")
try:
    # Using the same parameters as professor's code
    heu_net = pm4py.discover_heuristics_net(log, dependency_threshold=0.1)
    print("✓ Heuristic Miner completed successfully")
    
    # Visualize Heuristic Net
    pm4py.view_heuristics_net(heu_net)
    
    # Convert to BPMN
    bpmn_heuristic = pm4py.convert_to_bpmn(heu_net)
    pm4py.view_bpmn(bpmn_heuristic)
    print("✓ Heuristic Miner BPMN visualization opened")
    
except Exception as e:
    print(f"✗ Heuristic Miner failed: {e}")

# 4.3 Inductive Miner (using professor's approach)
print("\n--- INDUCTIVE MINER ---")
try:
    from pm4py.algo.discovery.inductive import algorithm as inductive_miner
    
    # Discover process tree using Inductive Miner (professor's exact function)
    tree = inductive_miner.apply(log)
    print("✓ Inductive Miner process tree discovered")
    
    # Convert process tree → BPMN (professor's exact approach)
    bpmn_inductive = pm4py.convert_to_bpmn(tree)
    
    # Visualize BPMN model
    pm4py.view_bpmn(bpmn_inductive)
    print("✓ Inductive Miner BPMN visualization opened")
    
    # Also show as Petri net for comparison
    im_net, im_initial_marking, im_final_marking = pm4py.convert_to_petri_net(tree)
    pm4py.view_petri_net(im_net, im_initial_marking, im_final_marking)
    
except Exception as e:
    print(f"✗ Inductive Miner failed: {e}")


=== PART 4: PROCESS DISCOVERY ALGORITHMS ===
Applying three process discovery algorithms...

--- ALPHA MINER ---
✓ Alpha+ Miner completed successfully
✗ Alpha+ Miner failed: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH
✓ Regular Alpha Miner completed successfully
✗ All Alpha variations failed: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

--- HEURISTIC MINER ---
✓ Heuristic Miner completed successfully
✗ Heuristic Miner failed: GraphViz's executables not found

--- INDUCTIVE MINER ---
✓ Inductive Miner process tree discovered
✗ Inductive Miner failed: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH
